In [ ]:
import requests
import zipfile
from pathlib import Path

def load_metadata(url: str, name: str, out: str = "/home/hmack/Development/rodent_experiments/datasets/",  ):

    dest = Path(out) / name

    zip_path = (dest).with_suffix(".zip")

    dest.mkdir(parents=True, exist_ok=True)

    with requests.get(url, stream = True) as r:
        r.raise_for_status()
        with open(zip_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024*1024):
                f.write(chunk)


    with zipfile.ZipFile(zip_path) as z:
        z.extractall(dest)

In [ ]:
import pandas as pd


## download metadata for AMMonitor Camera Traps


In [ ]:

import json
import pandas as pd
import requests
from pathlib import Path
from typing import Any
from tqdm.auto import tqdm

url = "https://lilawildlife.blob.core.windows.net/lila-wildlife/ammonitor-camera-traps/ammonitor-camera-traps.zip"
name = "ammonitor"

dest = "/home/hmack/Development/rodent_experiments/datasets"
load_metadata(url, name, out=dest)

def load_images(baseurl: str, out: str, metadata: pd.DataFrame, category: str, category_column: str = "category", filename_column:str = "file_name", map_category: str | None = None , limit: int | None = None, random_state:int = 42):

    # filter dataframe
    metadata_filtered = metadata.loc[metadata[category_column].str.strip().str.lower().isin([category,]), :]

    if map_category is None:
        map_category = category

    # make species directory
    outpath = Path(out) / map_category
    outpath.mkdir(parents=True, exist_ok=True)

    # remove duplicates from dataset
    metadata_filtered = metadata_filtered.drop_duplicates(subset = filename_column)

    if limit is not None:
        # randomize columns
        metadata_filtered = metadata_filtered.sample(frac=1, random_state=random_state)
    else:
        limit = int(1e9)

    # make category map to map dataset categories to species categories we want
    count = 0
    for idx, row in tqdm(metadata_filtered.iterrows(), total=len(metadata_filtered)):
        name = row[filename_column]
        img_bytes = requests.get(f"{baseurl}/{name}").content

        with open(Path(outpath) / f"{idx}.jpg", "wb") as f:
            f.write(img_bytes)

        if count >= limit:
            break

        count+=1

with open("/home/hmack/Development/rodent_experiments/datasets/ammonitor/ammonitor-camera-traps.json") as f:
    d = json.load(f)

images = pd.DataFrame(d["images"])        # id, file_name, location, datetime, dataset, seq_id, frame_num, seq_num_frames
annots = pd.DataFrame(d["annotations"])   # id, image_id, category_id
cats   = pd.DataFrame(d["categories"])    # id, name

df = annots.merge(images, left_on="image_id", right_on="id", suffixes=("_ann", "_img")).merge(cats.rename(columns={"id": "category_id", "name": "category"}), on="category_id")

used_category = list(filter(lambda x: any(f in x.lower() for f in ["mus", "mouse", "rat", "rattus", "domestic", "cat", "dog", "rodent", "murid", "shrew", "vole"]),  df.category.unique()))

relevant_categories = [
    "empty",
    "domestic cat",
    "domestic dog",
    "mouse sp.",
]

species_names = [
    "empty",
    "Felis catus",
    "Canis familiaris",
    "Mouse sp."
]

df_filtered = df.loc[df["category"].str.strip().str.lower().isin(relevant_categories), :]

for cat in relevant_categories:
    df_filtered = df.loc[df["category"].str.strip().str.lower().isin([cat,]), :]
    print(cat, ", ",   len(df_filtered))
metadata = df
baseurl = "https://storage.googleapis.com/public-datasets-lila/ammonitor-camera-traps"
output_path = "/home/hmack/Development/rodent_experiments/datasets/ammonitor_camera_traps"

for category, species_name in zip(relevant_categories, species_names):
    print(category, ", ", species_name)
    load_images(
        baseurl,
        output_path,
        metadata = metadata,
        category=category,
        map_category=species_name,
        limit=2000
    )

## download metadata for felidae-conservation-fund california dataset

In [ ]:
url = "https://lilawildlife.blob.core.windows.net/lila-wildlife/felidae-conservation-fund/felidae_conservation_fund_2020_2025.zip"
name = "felidae_conservation"
dest = "/home/hmack/Development/rodent_experiments/datasets"
load_metadata(url, name, out=dest)

In [ ]:
import json
import pandas as pd
metadata_path = "/home/hmack/Development/rodent_experiments/datasets/felidae_conservation/felidae_conservation_fund_2020_2025.json"

with open(metadata_path) as f:
    d = json.load(f)

images = pd.DataFrame(d["images"])        # id, file_name, location, datetime, dataset, seq_id, frame_num, seq_num_frames
annots = pd.DataFrame(d["annotations"])   # id, image_id, category_id
cats   = pd.DataFrame(d["categories"])    # id, name

df = annots.merge(images, left_on="image_id", right_on="id", suffixes=("_ann", "_img")).merge(cats.rename(columns={"id": "category_id", "name": "category"}), on="category_id")

sorted(df.category.unique())

In [ ]:
relevant_categories = list(filter(lambda x: any(f in x.lower() for f in ["rat", "mouse", "vole", "shrew", "mus", "rattus", "murid", "rodent"]), df.category.unique()))
relevant_categories

this dataset at least provides a 'mouse or rat' label, so it can be useful for the detection level

In [ ]:
for cat in relevant_categories:
    df_filtered = df.loc[df["category"].str.strip().str.lower().isin([cat,]), :]
    print(cat, ", ",   len(df_filtered))

# download metadata for wsu-lynx dataset 

In [ ]:
url = "https://lilawildlife.blob.core.windows.net/lila-wildlife/wsu-lynx/wsu-lynx.26.02.13.1705.zip"
name = "wsu_lynx"
dest = "/home/hmack/Development/rodent_experiments/datasets"
load_metadata(url, name, out=dest)

In [ ]:
import json
import pandas as pd
metadata_path = "/home/hmack/Development/rodent_experiments/datasets/wsu_lynx/wsu-lynx.json"

with open(metadata_path) as f:
    d = json.load(f)          # ~10-15 GB peak RAM; you have 42 GB free

images = pd.DataFrame(d["images"])        # id, file_name, location, datetime, dataset, seq_id, frame_num, seq_num_frames
annots = pd.DataFrame(d["annotations"])   # id, image_id, category_id
cats   = pd.DataFrame(d["categories"])    # id, name

df = annots.merge(images, left_on="image_id", right_on="id", suffixes=("_ann", "_img")).merge(cats.rename(columns={"id": "category_id", "name": "category"}), on="category_id")


sorted(df.category.unique())

In [ ]:
list(filter(lambda x: any(f in x.lower() for f in ["rat", "mouse", "vole", "shrew", "mus", "rattus", "murid", "rodent"]), df.category.unique()))

# download metadata for california small mammals dataset 

In [ ]:
url = "https://lilawildlife.blob.core.windows.net/lila-wildlife/california-small-animals/california_small_animals_with_sequences.zip"
name = "california_small_animals"
dest = "/home/hmack/Development/rodent_experiments/datasets"
load_metadata(url, name, out=dest)

In [ ]:
import json
import pandas as pd
metadata_path = "/home/hmack/Development/rodent_experiments/datasets/california_small_animals/california_small_animals_with_sequences.json"

with open(metadata_path) as f:
    d = json.load(f)          # ~10-15 GB peak RAM; you have 42 GB free

images = pd.DataFrame(d["images"])        # id, file_name, location, datetime, dataset, seq_id, frame_num, seq_num_frames
annots = pd.DataFrame(d["annotations"])   # id, image_id, category_id
cats   = pd.DataFrame(d["categories"])    # id, name

df = annots.merge(images, left_on="image_id", right_on="id", suffixes=("_ann", "_img")).merge(cats.rename(columns={"id": "category_id", "name": "category"}), on="category_id")


sorted(df.category.unique())

In [ ]:
relevant_categories = sorted(list(filter(lambda x: any(f.lower() in x.lower() for f in ["rat", "mouse", "vole", "shrew", "murid", "rodent", "sorex", "crocidura", "apodemus", "Myodes", "Arvicol", "mus", "rattus"]), df.category.unique())))
relevant_categories

In [ ]:
for cat in relevant_categories:
    df_filtered = df.loc[df["category"].str.strip().str.lower().isin([cat,]), :]
    print(cat, ", ",   len(df_filtered))

In [ ]:
used_categories = [
    "brown rat",
    "house mouse",
    "house rat",
    # "muridae family",
    # "mouse species",
    # "rattus species",
    # "rodent",
    "sorex species",
]

species_names = [
    "Rattus norvegicus",
    "Mus musculus",
    "Rattus rattus",
    # "Muridae",
    # "Rattus sp.",
    # "Rodentia",
    "Sorex sp."
]


for cat, sp in zip(used_categories, species_names):
    df_filtered = df.loc[df["category"].str.strip().str.lower().isin([cat,]), :]
    print(cat, ", ",   len(df_filtered))

In [ ]:
df_filtered = df.loc[df["category"].str.strip().str.lower().isin(used_categories), :]
df_filtered.loc[:, ["category", "class" ,"order", "family", "genus", "species", "file_name"]]

this dataset has sevearl interesting annotations which can be used, in particular shrews (shrew-mole, sorex) and house- and brown rat, and house mouse Snakes might also be interesting. Checking the providence is helpful, because everything from inaturalist we already have covered. 

In particular, various snakes seem to be useful 

In [ ]:
import requests
from pathlib import Path
from typing import Any
import pandas as pd
import json
import pandas as pd
from tqdm.auto import tqdm

def load_images(baseurl: str, out: str, metadata: pd.DataFrame, category: str, category_column: str = "category", filename_column:str = "file_name", map_category: str | None = None , limit: int | None = None, random_state:int = 42):

    # filter dataframe
    metadata_filtered = metadata.loc[metadata[category_column].str.strip().str.lower().isin([category,]), :]

    if map_category is None:
        map_category = category

    # make species directory
    outpath = Path(out) / map_category
    outpath.mkdir(parents=True, exist_ok=True)

    # remove duplicates from dataset
    metadata_filtered = metadata_filtered.drop_duplicates(subset = filename_column)

    if limit is not None:
        # randomize columns
        metadata_filtered = metadata_filtered.sample(frac=1, random_state=random_state)
    else:
        limit = int(1e9)

    # make category map to map dataset categories to species categories we want
    count = 0
    for idx, row in tqdm(metadata_filtered.iterrows(), total=len(metadata_filtered)):
        name = row[filename_column]
        img_bytes = requests.get(f"{baseurl}/{name}").content

        with open(Path(outpath) / f"{idx}.jpg", "wb") as f:
            f.write(img_bytes)

        if count >= limit:
            break

        count+=1


# make dataframe that contains metadata

metadata_path = "/home/hmack/Development/rodent_experiments/datasets/california_small_animals/california_small_animals_with_sequences.json"

with open(metadata_path) as f:
    d = json.load(f)          # ~10-15 GB peak RAM; you have 42 GB free

images = pd.DataFrame(d["images"])        # id, file_name, location, datetime, dataset, seq_id, frame_num, seq_num_frames
annots = pd.DataFrame(d["annotations"])   # id, image_id, category_id
cats   = pd.DataFrame(d["categories"])    # id, name

metadata = annots.merge(images, left_on="image_id", right_on="id", suffixes=("_ann", "_img")).merge(cats.rename(columns={"id": "category_id", "name": "category"}), on="category_id")

used_categories = [
    "brown rat",
    "house mouse",
    "house rat",
    # "muridae family",
    # "mouse species",
    # "rattus species",
    # "rodent",
    "shrew-mole",
    "sorex species",
]

species_names = [
    "Rattus norvegicus",
    "Mus musculus",
    "Rattus rattus",
    # "Muridae",
    # "Mus sp.",
    # "Rattus sp.",
    # "Rodentia",
    "Soricidae",
    "Sorex sp."
]

baseurl = "https://storage.googleapis.com/public-datasets-lila/california-small-animals"
output_path = "/home/hmack/Development/rodent_experiments/datasets/california_small_animals"

for category, species_name in zip(used_categories, species_names):
    print(category, ", ", species_name)
    load_images(
        baseurl,
        output_path,
        metadata = metadata,
        category=category,
        map_category=species_name
    )

# download metadata for ohio small animals dataset 

In [ ]:
url = "https://storage.googleapis.com/public-datasets-lila/osu-small-animals/osu-small-animals.json.zip"

name = "ohio_small_animals"
dest = "/home/hmack/Development/rodent_experiments/datasets"
load_metadata(url, name, out=dest)


In [ ]:
import json
import pandas as pd
metadata_path = "/home/hmack/Development/rodent_experiments/datasets/ohio_small_animals/osu-small-animals.json"

with open(metadata_path) as f:
    d = json.load(f)          # ~10-15 GB peak RAM; you have 42 GB free

images = pd.DataFrame(d["images"])        # id, file_name, location, datetime, dataset, seq_id, frame_num, seq_num_frames
annots = pd.DataFrame(d["annotations"])   # id, image_id, category_id
cats   = pd.DataFrame(d["categories"])    # id, name

df = annots.merge(images, left_on="image_id", right_on="id", suffixes=("_ann", "_img")).merge(cats.rename(columns={"id": "category_id", "name": "category"}), on="category_id")


sorted(df.category.unique())

In [ ]:
sorted(list(filter(lambda x: any(f.lower() in x.lower() for f in ["rat", "mouse", "vole", "shrew", "murid", "rodent", "sorex", "crocidura", "apodemus", "Myodes", "Arvicol", "mus", "rattus"]), df.category.unique())))

some labels seem useful, in particular some snakes seem to be there. 

In [ ]:
used_categories = [
    "brown_rat",
    "masked_shrew",
]

species_names = [
    "Rattus norvegicus",
    "Sorex cinereus"
]


for cat, sp in zip(used_categories, species_names):
    df_filtered = df.loc[df["category"].str.strip().str.lower().isin([cat,]), :]
    print(cat, ", ",   len(df_filtered))

In [ ]:
import requests
from pathlib import Path
from typing import Any
import pandas as pd
import json
import pandas as pd
from tqdm.auto import tqdm

def load_images(baseurl: str, out: str, metadata: pd.DataFrame, category: str, category_column: str = "category", filename_column:str = "file_name", map_category: str | None = None , limit: int | None = None, random_state:int = 42):

    # filter dataframe
    metadata_filtered = metadata.loc[metadata[category_column].str.strip().str.lower().isin([category,]), :]

    if map_category is None:
        map_category = category

    # make species directory
    outpath = Path(out) / map_category
    outpath.mkdir(parents=True, exist_ok=True)

    # remove duplicates from dataset
    metadata_filtered = metadata_filtered.drop_duplicates(subset = filename_column)

    if limit is not None:
        # randomize columns
        metadata_filtered = metadata_filtered.sample(frac=1, random_state=random_state)
    else:
        limit = int(1e9)

    # make category map to map dataset categories to species categories we want
    count = 0
    for idx, row in tqdm(metadata_filtered.iterrows(), total=len(metadata_filtered)):
        name = row[filename_column]
        img_bytes = requests.get(f"{baseurl}/{name}").content
        with open(Path(outpath) / f"{idx}.jpg", "wb") as f:
            f.write(img_bytes)

        if count >= limit:
            break

        count+=1

# make dataframe that contains metadata

metadata_path = "/home/hmack/Development/rodent_experiments/datasets/ohio_small_animals/osu-small-animals.json"

with open(metadata_path) as f:
    d = json.load(f)          # ~10-15 GB peak RAM; you have 42 GB free

images = pd.DataFrame(d["images"])        # id, file_name, location, datetime, dataset, seq_id, frame_num, seq_num_frames
annots = pd.DataFrame(d["annotations"])   # id, image_id, category_id
cats   = pd.DataFrame(d["categories"])    # id, name

metadata = annots.merge(images, left_on="image_id", right_on="id", suffixes=("_ann", "_img")).merge(cats.rename(columns={"id": "category_id", "name": "category"}), on="category_id")

used_categories = [
    "brown_rat",
    "masked_shrew",
]

species_names = [
    "Rattus norvegicus",
    "Sorex cinereus"
]

baseurl = "https://storage.googleapis.com/public-datasets-lila/osu-small-animals"
output_path = "/home/hmack/Development/rodent_experiments/datasets/ohio_small_animals"

for category, species_name in zip(used_categories, species_names):
    print(category, ", ", species_name)
    load_images(
        baseurl,
        output_path,
        metadata = metadata,
        category=category,
        map_category=species_name,
        limit = 1000
    )

## download labels for swg camera trap data 

In [ ]:
import json
import pandas as pd

url = "https://storage.googleapis.com/public-datasets-lila/swg-camera-traps/swg_camera_traps.zip"

name = "swg_camera_traps_2018-2020"
dest = "/home/hmack/Development/rodent_experiments/datasets"
load_metadata(url, name, out=dest)


metadata_path = "/home/hmack/Development/rodent_experiments/datasets/swg_camera_traps_2018-2020/swg_camera_traps.json"

with open(metadata_path) as f:
    d = json.load(f)          # ~10-15 GB peak RAM; you have 42 GB free

images = pd.DataFrame(d["images"])        # id, file_name, location, datetime, dataset, seq_id, frame_num, seq_num_frames
annots = pd.DataFrame(d["annotations"])   # id, image_id, category_id
cats   = pd.DataFrame(d["categories"])    # id, name

df = annots.merge(images, left_on="image_id", right_on="id", suffixes=("_ann", "_img")).merge(cats.rename(columns={"id": "category_id", "name": "category"}), on="category_id")

print(sorted(df.category.unique()))

used_categories = sorted(list(filter(lambda x: any(f.lower() in x.lower() for f in ["rat", "mouse", "vole", "shrew", "murid", "rodent", "sorex", "crocidura", "apodemus", "Myodes", "Arvicol", "mus", "rattus", "domestic", "human", "empty",  ]), df.category.unique())))


for cat in used_categories:
    df_filtered = df.loc[df["category"].str.strip().str.lower().isin([cat,]), :]
    print(cat, ", ",   len(df_filtered))

doesn´t seem particularly interesting, unless we want more 'murid' images

# download metadata for Idaho camera trap dataset 

In [ ]:
url = "https://storage.googleapis.com/public-datasets-lila/idaho-camera-traps/idaho-camera-traps.json.zip"

name = "idaho_camera_trap_data"
dest = "/home/hmack/Development/rodent_experiments/datasets"
load_metadata(url, name, out=dest)

In [ ]:
import json
import pandas as pd
metadata_path = "/home/hmack/Development/rodent_experiments/datasets/idaho_camera_trap_data/idaho-camera-traps.json"

with open(metadata_path) as f:
    d = json.load(f)          # ~10-15 GB peak RAM; you have 42 GB free

images = pd.DataFrame(d["images"])        # id, file_name, location, datetime, dataset, seq_id, frame_num, seq_num_frames
annots = pd.DataFrame(d["annotations"])   # id, image_id, category_id
cats   = pd.DataFrame(d["categories"])    # id, name

df = annots.merge(images, left_on="image_id", right_on="id", suffixes=("_ann", "_img")).merge(cats.rename(columns={"id": "category_id", "name": "category"}), on="category_id")

print(sorted(df.category.unique()))

used_categories = sorted(list(filter(lambda x: any(f.lower() in x.lower() for f in ["rat", "mouse", "vole", "shrew", "murid", "rodent", "sorex", "crocidura", "apodemus", "Myodes", "Arvicol", "mus", "rattus", "domestic", "human", "empty",  ]), df.category.unique())))


for cat in used_categories:
    df_filtered = df.loc[df["category"].str.strip().str.lower().isin([cat,]), :]
    print(cat, ", ",   len(df_filtered))

not good, unless we want various non-rodent images. 

# download metadata for North American Camera Trap dataset

In [ ]:
import pandas as pd

url = "https://storage.googleapis.com/public-datasets-lila/nacti/nacti_metadata.csv.zip"

name = "north_america_camera_trap_data"
dest = "/home/hmack/Development/rodent_experiments/datasets"
load_metadata(url, name, out=dest)

metadata_path = "/home/hmack/Development/rodent_experiments/datasets/north_america_camera_trap_data/nacti_metadata.csv"

df = pd.read_csv(metadata_path).dropna() # dorp the empty stuff

used_categories = sorted(list(filter(lambda x: any(f.lower() in x.lower() for f in ["rat", "mouse", "vole", "shrew", "murid", "rodent", "sorex", "crocidura", "apodemus", "Myodes", "Arvicol", "mus ", "rattus", "domestic", "human", "empty", "canidae", "homo",  ]), df.family.unique())))

actually_used_categories = ["canis familiaris", "muridae"]
for cat in actually_used_categories:
    df_filtered = df.loc[df["name"].str.strip().str.lower().isin([cat,]), :]
    print(cat, ", ",   len(df_filtered), ", ", df_filtered.loc[:, "name"].unique())
    print("########")

seems to be of limited use for our purposes. 

# caltech camera traps data


In [ ]:

import json
import pandas as pd
metadata_path = "/mnt/dataLinux/Development/rodent_experiments/datasets/caltech_camera_traps/caltech_images_20210113.json"

with open(metadata_path) as f:
    d = json.load(f)          # ~10-15 GB peak RAM; you have 42 GB free

images = pd.DataFrame(d["images"])        # id, file_name, location, datetime, dataset, seq_id, frame_num, seq_num_frames
annots = pd.DataFrame(d["annotations"])   # id, image_id, category_id
cats   = pd.DataFrame(d["categories"])    # id, name

df = annots.merge(images, left_on="image_id", right_on="id", suffixes=("_ann", "_img")).merge(cats.rename(columns={"id": "category_id", "name": "category"}), on="category_id")

print(sorted(df.category.unique()))

used_categories = sorted(list(filter(lambda x: any(f.lower() in x.lower() for f in ["rat", "mouse", "vole", "shrew", "murid", "rodent", "sorex", "crocidura", "apodemus", "Myodes", "Arvicol", "mus", "rattus", "domestic", "human", "empty", "car", "cow", "pig"]), df.category.unique())))


for cat in used_categories:
    df_filtered = df.loc[df["category"].str.strip().str.lower().isin([cat,]), :]
    print(cat, ", ",   len(df_filtered))

# explore caltech camera trap data

In [ ]:
import pandas as pd
import json

imagejson = "/mnt/dataLinux/Development/rodent_experiments/datasets/caltech_camera_traps/caltech_images_20210113.json"
bboxjson = "/mnt/dataLinux/Development/rodent_experiments/datasets/caltech_camera_traps/caltech_bboxes_20200316.json"

with open(imagejson, "r") as file:
    images = pd.DataFrame(json.load(file)["images"])
    images = images.rename(
        columns = {
            "id": "image_id"
        }
    )
with open(bboxjson, "r") as file:
    bboxes = json.load(file)
    categories = pd.DataFrame(bboxes["categories"])
    annotations = pd.DataFrame(bboxes["annotations"])


print(len(images), len(annotations))


# make dataframes and join on image_id
caltech_data = images.join(
    annotations.set_index("image_id"),
    on = "image_id",
    how="inner"
)

caltech_data.to_csv("/mnt/dataLinux/Development/rodent_experiments/datasets/caltech_camera_traps/caltech_data.csv", index=False)

In [ ]:
caltech_data.columns

## filter caltech data for interesting stuff

... and put that into the dataset we need

In [ ]:
import pandas as pd
from pathlib import Path
from shutil import copy2
from tqdm import tqdm
caltech_data = pd.read_csv(
    "/mnt/dataLinux/Development/rodent_experiments/datasets/caltech_camera_traps/caltech_data.csv"
)

# category_ids:
catmap = {
    30: "empty",
    33: "car",
    39: "Sus domesticus",
    37: "Bos taurus",
    16: "Felis catus",
    8: "Canis familiaris",
}

caltech_data_filtered = caltech_data[caltech_data.category_id.isin([30, 33, 39, 37, 16, 8])]
caltech_data_filtered["category_id"] = caltech_data_filtered["category_id"].map(catmap)


image_path = Path("/mnt/dataLinux/Development/rodent_experiments/datasets/caltech_camera_traps/cct_images")
out_path = Path("/mnt/dataLinux/Development/rodent_experiments/datasets/biotrove-central-europe")

for row in tqdm(caltech_data_filtered.itertuples(), total = len(caltech_data_filtered)):
    cat = str(row.category_id)
    filename = str(row.file_name)
    catpath = Path(out_path) / cat
    if catpath.exists() is False:
        catpath.mkdir(parents=True)

    copy2(
        str(image_path / filename),
        str(catpath / filename)
    )


## Download inaturalist datasets 

In [ ]:
from smartrodent import inaturalist

config_path = "/home/hmack/Development/rodent_experiments/configs/data_config_central_europe.yaml"

inaturalist.download_inat_data(
    config_path
)